In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import pandas as pd
import re
import json
import os
from tqdm import tqdm

In [15]:
input_csv = "/content/drive/MyDrive/Penalaran Komputer 202310370311281/tugas subcpmk3/PenalaranKomputer_CBR/03_Dataset_CSV/cases.csv"

output_csv = "/content/drive/MyDrive/Penalaran Komputer 202310370311281/tugas subcpmk3/PenalaranKomputer_CBR/04_Case_Representation/cases_representation.csv"

output_json = "/content/drive/MyDrive/Penalaran Komputer 202310370311281/tugas subcpmk3/PenalaranKomputer_CBR/04_Case_Representation/cases_representation.json"

In [5]:
input_folder = "/content/drive/MyDrive/Penalaran Komputer 202310370311281/tugas subcpmk3/PenalaranKomputer_CBR/02_Dataset_Text_Final"

output_csv = "/content/drive/MyDrive/Penalaran Komputer 202310370311281/tugas subcpmk3/PenalaranKomputer_CBR/04_Case_Representation/cases_representation.csv"

In [16]:
df = pd.read_csv(input_csv)

print(df.shape)

df.head()

(40, 3)


,id,filename,case_text
0,1,putusan_001.txt,gNomor 431 K/Pid/2026 e DEMI KEADILAN BERDASAR...
1,2,putusan_002.txt,Nomor 1972 K/Pid/2025 n DEMI KnEADILAN BERDASA...
2,3,putusan_003.txt,gNomor 1805 K/Pid/2025 e DEMI KEADILAN BERDASA...
3,4,putusan_004.txt,gNomor 1593 K/Pid/2025 e DEMI KEADILAN BERDASA...
4,5,putusan_005.txt,gNomor 1488 K/Pid/2025 e DEMI KEADILAN BERDASA...


In [18]:
def extract_no_perkara(text):

    match = re.search(
        r'Nomor\s+([\d]+[\w\s\/\.-]*)',
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    return "Tidak Ditemukan"


def extract_tanggal_putusan(text):

    pattern = r'(\d{1,2}\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+\d{4})'

    matches = re.findall(pattern, text)

    if matches:
        return matches[-1][0]

    return "Tidak Ditemukan"


def extract_pasal(text):

    hasil = re.findall(
        r'Pasal\s+\d+\s*(?:Ayat\s*\(\d+\))?\s*[A-Za-z\.]*',
        text,
        re.IGNORECASE
    )

    hasil = list(set(hasil))

    return "; ".join(hasil[:10])


def extract_pihak(text):

    pihak = []

    if "Penuntut Umum" in text:
        pihak.append("Penuntut Umum")

    if "Terdakwa" in text:
        pihak.append("Terdakwa")

    if "Pemohon Kasasi" in text:
        pihak.append("Pemohon Kasasi")

    return "; ".join(pihak)

In [19]:
def extract_ringkasan_fakta(text):

    kata_kunci = [
        "didakwa",
        "dakwaan",
        "barang bukti",
        "terdakwa diajukan"
    ]

    for kata in kata_kunci:

        idx = text.lower().find(kata)

        if idx != -1:

            return text[idx:idx+800]

    return text[:800]

In [20]:
def extract_argumen_hukum(text):

    kata_kunci = [
        "menimbang bahwa",
        "mahkamah agung berpendapat",
        "berdasarkan pertimbangan tersebut"
    ]

    lower = text.lower()

    for kata in kata_kunci:

        idx = lower.find(kata)

        if idx != -1:

            return text[idx:idx+1200]

    return ""

In [21]:
rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    text = str(row["case_text"])

    data = {

        "case_id": row["id"],

        "filename": row["filename"],

        "no_perkara": extract_no_perkara(text),

        "tanggal": extract_tanggal_putusan(text),

        "pasal": extract_pasal(text),

        "pihak": extract_pihak(text),

        "ringkasan_fakta": extract_ringkasan_fakta(text),

        "argumen_hukum": extract_argumen_hukum(text),

        "length": len(text.split()),

        "text_full": text
    }

    rows.append(data)

case_df = pd.DataFrame(rows)

case_df.head()

100%|██████████| 40/40 [00:00<00:00, 490.23it/s]


,case_id,filename,no_perkara,tanggal,pasal,pihak,ringkasan_fakta,argumen_hukum,length,text_full
0,1,putusan_001.txt,431 K/Pid/2026 e DEMI KEADILAN BERDASARKAN KET...,7 April 2026,Pasal 75 Undang; Pasal 433 a; Pasal 433 Ayat (...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: d g Ke...,Menimbang bahwa Putusan Pengadilan Tinggi Meda...,1994,gNomor 431 K/Pid/2026 e DEMI KEADILAN BERDASAR...
1,2,putusan_002.txt,1972 K/Pid/2025 n DEMI KnEADILAN BERDASARKAN K...,27 November 2025,Pasal 311 Ayat (1) KUHP; Pasal 253 Ayat (1) Un...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: d g Pe...,Menimbang bahwa Putusan Pengadilan Tinggi Kupa...,2588,Nomor 1972 K/Pid/2025 n DEMI KnEADILAN BERDASA...
2,3,putusan_003.txt,1805 K/Pid/2025 e DEMI KEADILAN BERDASARKAN KE...,24 Oktober 2025,Pasal 317 Ayat (1) KUHnP; Pasal 253 KUHAP; Pas...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: d g Pe...,Menimbang bahwa Putusan Pengadilan Tinggi Sura...,3112,gNomor 1805 K/Pid/2025 e DEMI KEADILAN BERDASA...
3,4,putusan_004.txt,1593 K/Pid/2025 e DEMI KEADILAN BERDASARKAN KE...,19 September 2025,Pasal 311 Ayat (1) KUHP; Pasal 310 Ayat (1) KU...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: o u Pe...,Menimbang bahwa Peengadilan Tinggi Jawa Tengah...,1722,gNomor 1593 K/Pid/2025 e DEMI KEADILAN BERDASA...
4,5,putusan_005.txt,1488 K/Pid/2025 e DEMI KEADILAN BERDASARKAN KE...,19 September 2025,Pasal 253 Ayat (1) KUHAP; Pasal 315 KUHP; Pasa...,Penuntut Umum; Terdakwa; Pemohon Kasasi,"didakwa dengan dakwaan Tunggal, sebagaimana di...",Menimbang bahwa putusan Pengadilan Tinggi Meda...,1537,gNomor 1488 K/Pid/2025 e DEMI KEADILAN BERDASA...


In [22]:
case_df.to_csv(
    output_csv,
    index=False
)

case_df.to_json(
    output_json,
    orient="records",
    force_ascii=False,
    indent=4
)

print("Selesai")
print(output_csv)
print(output_json)

Selesai
/content/drive/MyDrive/Penalaran Komputer 202310370311281/tugas subcpmk3/PenalaranKomputer_CBR/04_Case_Representation/cases_representation.csv
/content/drive/MyDrive/Penalaran Komputer 202310370311281/tugas subcpmk3/PenalaranKomputer_CBR/04_Case_Representation/cases_representation.json


In [24]:
case_df.head()

,case_id,filename,no_perkara,tanggal,pasal,pihak,ringkasan_fakta,argumen_hukum,length,text_full
0,1,putusan_001.txt,431 K/Pid/2026 e DEMI KEADILAN BERDASARKAN KET...,7 April 2026,Pasal 75 Undang; Pasal 433 a; Pasal 433 Ayat (...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: d g Ke...,Menimbang bahwa Putusan Pengadilan Tinggi Meda...,1994,gNomor 431 K/Pid/2026 e DEMI KEADILAN BERDASAR...
1,2,putusan_002.txt,1972 K/Pid/2025 n DEMI KnEADILAN BERDASARKAN K...,27 November 2025,Pasal 311 Ayat (1) KUHP; Pasal 253 Ayat (1) Un...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: d g Pe...,Menimbang bahwa Putusan Pengadilan Tinggi Kupa...,2588,Nomor 1972 K/Pid/2025 n DEMI KnEADILAN BERDASA...
2,3,putusan_003.txt,1805 K/Pid/2025 e DEMI KEADILAN BERDASARKAN KE...,24 Oktober 2025,Pasal 317 Ayat (1) KUHnP; Pasal 253 KUHAP; Pas...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: d g Pe...,Menimbang bahwa Putusan Pengadilan Tinggi Sura...,3112,gNomor 1805 K/Pid/2025 e DEMI KEADILAN BERDASA...
3,4,putusan_004.txt,1593 K/Pid/2025 e DEMI KEADILAN BERDASARKAN KE...,19 September 2025,Pasal 311 Ayat (1) KUHP; Pasal 310 Ayat (1) KU...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: o u Pe...,Menimbang bahwa Peengadilan Tinggi Jawa Tengah...,1722,gNomor 1593 K/Pid/2025 e DEMI KEADILAN BERDASA...
4,5,putusan_005.txt,1488 K/Pid/2025 e DEMI KEADILAN BERDASARKAN KE...,19 September 2025,Pasal 253 Ayat (1) KUHAP; Pasal 315 KUHP; Pasa...,Penuntut Umum; Terdakwa; Pemohon Kasasi,"didakwa dengan dakwaan Tunggal, sebagaimana di...",Menimbang bahwa putusan Pengadilan Tinggi Meda...,1537,gNomor 1488 K/Pid/2025 e DEMI KEADILAN BERDASA...
